## Aim:

This notebook takes a directory of DEM tiles (imported from Copernicus Data Explorer, or DEFRA (Uk.gov)), and exports a merged DEM .tif.

AI helped, but did NOT generate, parts of the logic in this code. Specifically, it helped with the tempfile functionality, as this was new to me.

This code has been run every time I need to merge Digital Elevation Model Tiles into one raster.

In [11]:
import rasterio
from rasterio.merge import merge
from pathlib import Path
import numpy as np
import glob

dem_files = [Path('../../data/HillShade/BCCheddar/out/preds/mosaic.tif'), 
            Path('../../data/HillShade/out/preds/mosaic.tif'),
            Path('../../data/HillShade/HillShadeExtension/out/preds/mosaic.tif'),
            Path('../../data/HillShade/BCEast2/out/preds/mosaic.tif')]

OUT_PATH = Path('../../data/HillShade/out/preds')

In [12]:
dem_files

[PosixPath('../../data/HillShade/BCCheddar/out/preds/mosaic.tif'),
 PosixPath('../../data/HillShade/out/preds/mosaic.tif'),
 PosixPath('../../data/HillShade/HillShadeExtension/out/preds/mosaic.tif'),
 PosixPath('../../data/HillShade/BCEast2/out/preds/mosaic.tif')]

In [ ]:
import tempfile, os
import numpy as np
import rasterio
from rasterio.merge import merge

# handles -1e38 in DEm tile sbeing nodata
NODATA = -9999
clipped_paths = []
tmpdir = tempfile.mkdtemp()

for f in dem_files:
    with rasterio.open(f) as src:
        data = src.read(1).astype('float32')

        if src.nodata is not None:
            data[data == src.nodata] = NODATA

        profile = src.profile.copy()
        profile.update(dtype='float32', nodata=NODATA)

        out_name = f"{f.parent.parent.parent.name}_{f.name}" 
        out_path = os.path.join(tmpdir, out_name)
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data, 1)

        clipped_paths.append(out_path)

# Merge
datasets = [rasterio.open(p) for p in clipped_paths]
mosaic, transform = merge(datasets, nodata=NODATA, method='max') 

profile = datasets[0].profile
profile.update(
    width=mosaic.shape[2],
    height=mosaic.shape[1],
    transform=transform,
    dtype='float32',
    nodata=NODATA,
)

with rasterio.open(OUT_PATH / 'FullMosaic.tif', 'w', **profile) as dst:
    dst.write(mosaic)

for ds in datasets:
    ds.close()
print("Saved")

Saved
